In [ ]:
# 7 giugno: canali piu' discriminativi ricavati dagli ERD/ERS.
# La logica di calcolo vive in util/channel_selection.py, cosi' la stessa funzione viene
# richiamata dentro le pipeline. Qui il notebook serve solo a ispezionare la selezione.
#
# Importante: i canali vengono stampati per ogni fold della Leave-One-Run-Out, calcolati
# sulle sole run di training di quel fold. Una selezione unica calcolata su tutte e tre le
# run userebbe anche la run di test, falsando i risultati della validazione.
# --- Preambolo standard ------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import mne
from util.channel_selection import select_discriminative_channels, IMAGERY_REST_ACTIVE

mne.set_log_level('ERROR')

root = DATA
runs = ["4", "8", "12"]
first_person = 1
people = 10

N_CHANNELS = 10
CLASS_MAP = IMAGERY_REST_ACTIVE   # in alternativa: IMAGERY_LEFT_RIGHT, EXECUTION_*

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    print(f"Paziente {subject}")

    for test_run in runs:
        train_runs = [r for r in runs if r != test_run]

        try:
            channels = select_discriminative_channels(
                subject,
                train_runs,
                root,
                class_map=CLASS_MAP,
                n_channels=N_CHANNELS,
            )
            print(f'  test={test_run} (train {"+".join(train_runs)}): {channels}')

        except Exception as e:
            print(f"  test={test_run}: errore - {e}")

    print()
